In [1]:
import subprocess
import sys

packages = ["diffusers", "transformers", "accelerate", "safetensors", "pandas", "pillow", "matplotlib"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])

0

In [2]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

project_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "notebooks").is_dir() and (path / "src").is_dir()),
    Path.cwd()
)
DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports" / "day15_marketing_images"
SRC_DIR = PROJECT_ROOT / "src"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())

Project: c:\Users\rupam\OneDrive\Desktop\Retail_Mind
CUDA available: False


In [3]:
csv_candidates = list(DATA_DIR.rglob("styles.csv")) if DATA_DIR.exists() else []

if not csv_candidates:
    raise FileNotFoundError(
        "styles.csv was not found. Place your existing Fashion Product Images dataset under data/."
    )

STYLES_CSV = csv_candidates[0]
styles_df = pd.read_csv(STYLES_CSV, on_bad_lines="skip", encoding_errors="replace")
styles_df.columns = [str(column).strip() for column in styles_df.columns]

required_columns = {"masterCategory", "subCategory", "articleType", "baseColour", "productDisplayName"}
missing_columns = required_columns.difference(styles_df.columns)
if missing_columns:
    raise ValueError(f"styles.csv is missing columns: {sorted(missing_columns)}")

styles_df = styles_df.dropna(subset=["articleType", "productDisplayName"]).reset_index(drop=True)
PRODUCT_ID_COLUMN = "id" if "id" in styles_df.columns else styles_df.columns[0]
PRODUCT_ID = None

if PRODUCT_ID is None:
    product_row = styles_df.iloc[0]
else:
    matched_rows = styles_df[styles_df[PRODUCT_ID_COLUMN].astype(str) == str(PRODUCT_ID)]
    if matched_rows.empty:
        raise ValueError(f"Product ID {PRODUCT_ID} was not found in styles.csv")
    product_row = matched_rows.iloc[0]

product_metadata = product_row.to_dict()
print("Dataset:", STYLES_CSV)
print("Products:", len(styles_df))
display(pd.DataFrame([product_metadata]))

Dataset: c:\Users\rupam\OneDrive\Desktop\Retail_Mind\data\fashion_product_images_small\styles.csv
Products: 44417


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt


In [4]:
from diffusers import StableDiffusionPipeline

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

if DEVICE == "cpu":
    print("CUDA GPU was not detected. Generation will run on CPU and may be very slow.")

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    use_safetensors=True
)
pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

print("Loaded:", MODEL_ID)
print("Device:", DEVICE)

c:\Users\rupam\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA GPU was not detected. Generation will run on CPU and may be very slow.


c:\Users\rupam\AppData\Local\Programs\Python\Python314\Lib\site-packages\diffusers\utils\deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
c:\Users\rupam\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rupam\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-v1-5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Dev

RuntimeError: Task error: File reconstruction error: IO Error: There is not enough space on the disk. (os error 112)

In [ ]:
BRAND_STYLE = "premium modern retail campaign, clean composition, elegant neutral color palette, realistic commercial product photography"

PROMPT_TEMPLATES = {
    "clean_catalog": "Professional ecommerce catalog photograph of {product}, category {category}, subcategory {subcategory}, article type {article_type}, colour {colour}. Product centered, fully visible, clean seamless studio background, soft balanced studio lighting, accurate material texture, sharp details, premium retail photography.",
    "premium_campaign": "Premium fashion advertising campaign featuring {product}, category {category}, article type {article_type}, colour {colour}. Editorial product photography, refined composition, subtle cinematic lighting, sophisticated styling, polished commercial campaign aesthetic.",
    "seasonal_lifestyle": "Lifestyle fashion campaign featuring {product}, category {category}, article type {article_type}, colour {colour}, season {season}, usage {usage}. Natural lifestyle setting, tasteful props, realistic lighting, aspirational retail advertising photography."
}

DEFAULT_NEGATIVE_PROMPT = "low quality, blurry, pixelated, distorted product, incorrect shape, duplicate product, extra objects, cropped product, out of frame, messy background, poor lighting, oversaturated, text, typography, letters, watermark, logo, brand mark, price tag, label, signature"


def metadata_value(metadata, key, fallback="not specified"):
    value = metadata.get(key, fallback)
    if pd.isna(value) or str(value).strip() == "":
        return fallback
    return str(value).strip()


def build_product_prompt(metadata, template_name="clean_catalog", brand_style=BRAND_STYLE, setting=None):
    if template_name not in PROMPT_TEMPLATES:
        raise ValueError(f"Unknown template: {template_name}")

    values = {
        "product": metadata_value(metadata, "productDisplayName", "fashion product"),
        "category": metadata_value(metadata, "masterCategory"),
        "subcategory": metadata_value(metadata, "subCategory"),
        "article_type": metadata_value(metadata, "articleType"),
        "colour": metadata_value(metadata, "baseColour"),
        "season": metadata_value(metadata, "season"),
        "usage": metadata_value(metadata, "usage")
    }
    prompt = PROMPT_TEMPLATES[template_name].format(**values)
    if setting:
        prompt += f" Setting: {setting}."
    return f"{prompt} {brand_style}. Do not add text or logos."

In [ ]:
TEMPLATE_NAME = "clean_catalog"
SETTING = "a simple warm off-white studio backdrop"
PROMPT = build_product_prompt(product_metadata, TEMPLATE_NAME, setting=SETTING)
NEGATIVE_PROMPT = DEFAULT_NEGATIVE_PROMPT

GUIDANCE_SCALE = 7.5
NUM_INFERENCE_STEPS = 30
IMAGE_WIDTH = 512
IMAGE_HEIGHT = 512

print(PROMPT)
print("\nNegative prompt:", NEGATIVE_PROMPT)

In [ ]:
product_id = str(product_metadata.get(PRODUCT_ID_COLUMN, "product"))
output_path = REPORT_DIR / f"product_{product_id}_{TEMPLATE_NAME}_seed_{SEED}.png"
generator = torch.Generator(device=DEVICE).manual_seed(SEED)

with torch.inference_mode():
    generated_image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        width=IMAGE_WIDTH,
        height=IMAGE_HEIGHT,
        generator=generator
    ).images[0]

generated_image.save(output_path)
print("Saved:", output_path)

In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(generated_image)
plt.axis("off")
plt.title(f"RetailMind Marketing Image | Product {product_id}")
plt.show()

In [ ]:
prompt_library = {}
for template_name in PROMPT_TEMPLATES:
    prompt_library[template_name] = {
        "product_id": product_id,
        "prompt": build_product_prompt(product_metadata, template_name, setting=SETTING),
        "negative_prompt": NEGATIVE_PROMPT,
        "guidance_scale": GUIDANCE_SCALE,
        "steps": NUM_INFERENCE_STEPS,
        "seed": SEED
    }

prompt_library_path = REPORT_DIR / "prompt_library.json"
with open(prompt_library_path, "w", encoding="utf-8") as file:
    json.dump(prompt_library, file, indent=2, ensure_ascii=False)

print("Prompt library saved:", prompt_library_path)
display(pd.DataFrame([
    {"template": name, "prompt": value["prompt"]}
    for name, value in prompt_library.items()
]))

In [ ]:
module_code = r'''import json
from pathlib import Path

import torch

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
BRAND_STYLE = "premium modern retail campaign, clean composition, elegant neutral color palette, realistic commercial product photography"
PROMPT_TEMPLATES = {
    "clean_catalog": "Professional ecommerce catalog photograph of {product}, category {category}, subcategory {subcategory}, article type {article_type}, colour {colour}. Product centered, fully visible, clean seamless studio background, soft balanced studio lighting, accurate material texture, sharp details, premium retail photography.",
    "premium_campaign": "Premium fashion advertising campaign featuring {product}, category {category}, article type {article_type}, colour {colour}. Editorial product photography, refined composition, subtle cinematic lighting, sophisticated styling, polished commercial campaign aesthetic.",
    "seasonal_lifestyle": "Lifestyle fashion campaign featuring {product}, category {category}, article type {article_type}, colour {colour}, season {season}, usage {usage}. Natural lifestyle setting, tasteful props, realistic lighting, aspirational retail advertising photography."
}
DEFAULT_NEGATIVE_PROMPT = "low quality, blurry, pixelated, distorted product, incorrect shape, duplicate product, extra objects, cropped product, out of frame, messy background, poor lighting, oversaturated, text, typography, letters, watermark, logo, brand mark, price tag, label, signature"


def _value(metadata, key, fallback="not specified"):
    value = metadata.get(key, fallback)
    if value is None or str(value).strip().lower() in {"", "nan", "none"}:
        return fallback
    return str(value).strip()


def build_product_prompt(metadata, template_name="clean_catalog", brand_style=BRAND_STYLE, setting=None):
    if template_name not in PROMPT_TEMPLATES:
        raise ValueError(f"Unknown template: {template_name}")
    values = {
        "product": _value(metadata, "productDisplayName", "fashion product"),
        "category": _value(metadata, "masterCategory"),
        "subcategory": _value(metadata, "subCategory"),
        "article_type": _value(metadata, "articleType"),
        "colour": _value(metadata, "baseColour"),
        "season": _value(metadata, "season"),
        "usage": _value(metadata, "usage")
    }
    prompt = PROMPT_TEMPLATES[template_name].format(**values)
    if setting:
        prompt += f" Setting: {setting}."
    return f"{prompt} {brand_style}. Do not add text or logos."


def load_pipeline(model_id=MODEL_ID, device=None):
    from diffusers import StableDiffusionPipeline
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if device == "cuda" else torch.float32
    pipeline = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=dtype,
        use_safetensors=True
    )
    pipeline = pipeline.to(device)
    pipeline.enable_attention_slicing()
    pipeline.enable_vae_slicing()
    return pipeline, device


def generate_marketing_image(
    pipeline,
    metadata,
    output_path,
    template_name="clean_catalog",
    setting=None,
    brand_style=BRAND_STYLE,
    negative_prompt=DEFAULT_NEGATIVE_PROMPT,
    seed=42,
    steps=30,
    guidance_scale=7.5,
    width=512,
    height=512,
    device=None
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    prompt = build_product_prompt(metadata, template_name, brand_style, setting)
    generator = torch.Generator(device=device).manual_seed(int(seed))
    with torch.inference_mode():
        image = pipeline(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=int(steps),
            guidance_scale=float(guidance_scale),
            width=int(width),
            height=int(height),
            generator=generator
        ).images[0]
    destination = Path(output_path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    image.save(destination)
    return {"image": image, "prompt": prompt, "negative_prompt": negative_prompt, "path": str(destination), "seed": int(seed)}


def generate_from_product_id(
    pipeline,
    product_id,
    styles_csv_path,
    output_path,
    template_name="clean_catalog",
    **generation_options
):
    import pandas as pd
    products = pd.read_csv(styles_csv_path, on_bad_lines="skip", encoding_errors="replace")
    id_column = "id" if "id" in products.columns else products.columns[0]
    selected = products[products[id_column].astype(str) == str(product_id)]
    if selected.empty:
        raise ValueError(f"Product ID {product_id} was not found")
    metadata = selected.iloc[0].to_dict()
    return generate_marketing_image(
        pipeline,
        metadata,
        output_path,
        template_name=template_name,
        **generation_options
    )
'''

module_path = SRC_DIR / "marketing_image_generator.py"
module_path.write_text(module_code, encoding="utf-8")
print("Module saved:", module_path)

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.marketing_image_generator import generate_marketing_image

module_result = generate_marketing_image(
    pipeline=pipe,
    metadata=product_metadata,
    output_path=REPORT_DIR / f"module_test_product_{product_id}_seed_{SEED}.png",
    template_name="clean_catalog",
    setting=SETTING,
    negative_prompt=NEGATIVE_PROMPT,
    seed=SEED,
    steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    width=IMAGE_WIDTH,
    height=IMAGE_HEIGHT,
    device=DEVICE
)

print(module_result["path"])
print(module_result["prompt"])